# Tutorial 1: Learn about `requests`

Follow along with [this tutorial on RealPython](https://realpython.com/python-requests/). 
Complete the first 5 sections thoroughly (up to but not including User Other HTTP methods). Then jump to section Improve Performance to learn about some advanced tricks that you might need at some point when scraping websites that have a lot of content.

Use markdown headings as appropriate to enumerate sections.

# Python's Requests Library (Guide)

## Get Started With Python’s Requests Library

In [ ]:
import requests

## Make a GET Request

In [ ]:
requests.get("https://api.github.com")

## Inspect the Response

In [ ]:
response = requests.get("https://api.github.com")

### Work With Status Codes

In [ ]:
response.status_code

In [ ]:
if response.status_code == 200:
    print("Success!")
elif response.status_code == 404:
    print("Not Found.")

In [ ]:
if response:
    print("Success!")
else:
    raise Exception(f"Non-success status code: {response.status_code}")

In [ ]:
import requests
from requests.exceptions import HTTPError

URLS = ["https://api.github.com", "https://api.github.com/invalid"]

for url in URLS:
    try:
        response = requests.get(url)
        response.raise_for_status()
    except HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
    except Exception as err:
        print(f"Other error occurred: {err}")
    else:
        print("Success!")

### Access the Response Content

In [ ]:
response.content

In [ ]:
type(response.content)

In [ ]:
response.text

In [ ]:
type(response.text)

In [ ]:
response.encoding = "utf-8"
response.text

In [ ]:
response.json()

In [ ]:
type(response.json())

In [ ]:
response_dict = response.json()
response_dict["emojis_url"]

### View Response Headers

In [ ]:
response.headers

In [ ]:
response.headers["Content-Type"]

In [ ]:
response.headers["content-type"]

## Add Query String Parameters

In [ ]:
response = requests.get(
    "https://api.github.com/search/repositories",
    params={"q": "language:python", "sort": "stars", "order": "desc"},
)

json_response = response.json()
popular_repositories = json_response["items"]
for repo in popular_repositories[:3]:
    print(f"Name: {repo['name']}")
    print(f"Description: {repo['description']}")
    print(f"Stars: {repo['stargazers_count']}\n")

In [ ]:
requests.get(
...     "https://api.github.com/search/repositories",
...     [("q", "language:python"), ("sort", "stars"), ("order", "desc")],
... )

In [ ]:
requests.get(
...     "https://api.github.com/search/repositories",
...     params=b"q=language:python&sort=stars&order=desc",
... )

## Customize Request Headers

In [ ]:
response = requests.get(
    "https://api.github.com/search/repositories",
    params={"q": '"real python"'},
    headers={"Accept": "application/vnd.github.text-match+json"},
)

json_response = response.json()
first_repository = json_response["items"][0]
print(first_repository["text_matches"][0]["matches"])

## Improve Performance

### Set Request Timeouts

In [ ]:
requests.get("https://api.github.com", timeout=1)

In [ ]:
requests.get("https://api.github.com", timeout=(3.05, 5))

In [ ]:
from requests.exceptions import Timeout

try:
    response = requests.get("https://api.github.com", timeout=(3.05, 5))
except Timeout:
    print("The request timed out")
else:
    print("The request did not time out")

### Reuse Connections With Session Objects

In [ ]:
from custom_token_auth import TokenAuth

TOKEN = "<YOUR_GITHUB_PA_TOKEN>"

with requests.Session() as session:
    session.auth = TokenAuth(TOKEN)

    first_response = session.get("https://api.github.com/user")
    second_response = session.get("https://api.github.com/user")

print(first_response.headers)
print(second_response.json())

### Retry Failed Requests

In [ ]:
from requests.adapters import HTTPAdapter
from requests.exceptions import RetryError
from urllib3.util.retry import Retry

retry_strategy = Retry(
    total=2,
    status_forcelist=[429, 500, 502, 503, 504]
)
github_adapter = HTTPAdapter(max_retries=retry_strategy)

with requests.Session() as session:
    session.mount("https://api.github.com", github_adapter)
    try:
        response = session.get("https://api.github.com/")
    except RetryError as err:
        print(f"Error: {err}")